In [27]:
import os
import sys
import glob
import multiprocessing
import subprocess
import time
from datetime import datetime as dt
from os import path
from IPython.display import clear_output

In [28]:
#input parameters
problems_name_dir: str = "../problems/benchmark/**/*.gz"
min_dim: int = 100 #minimal problem dim
max_dim: int = 10000 #maximal problem dim
submat_size_lim: int = 200 #maximal size of square submat to generate conic cover cuts
row_trials_num: int = 7 #number of row trials to generate cuts
cuts_gen_rounds: int = 1 #number of cuts generation iterations
model_cuts_switch: str = "off" #switch for model cuts option
node_lim: int = 0 #node limit for CPlex
available_cpu_cores: int = 30

In [29]:
#internal parameters and constants
cplex_workers: int = 15
cuts_gen_tl: int = 0 #cuts gen time limit in minutes
cplex_tl: int = 180 #cplex time limit in minutes

log_file_name_suff: str = "_log.txt" #out log file name suffix
result_file_name_suff: str = "_results.txt" #results file name suffix
cuts_file_name_suff: str = "_cuts.txt" #out cuts file name

out_dir_pref: str = "cutsPower__"
run_path: str = "../build/" 

cuts_gen_sec_tl: int = cuts_gen_tl * 60 #cuts gen time limit in seconds
cplex_sec_tl: int = cplex_tl * 60 #cplex time limit in seconds

In [30]:
def form_out_dir_name():
    time_now: str = dt.now().strftime("%Y.%m.%d_%H-%M-%S")
    return out_dir_pref + time_now

out_dir_name: str = form_out_dir_name()
print(f"Output directory name: {out_dir_name}")

os.mkdir(run_path+out_dir_name)
print(f"The outpud directory ({run_path+out_dir_name}) is created!")

Output directory name: cutsPower__2023.08.18_06-59-25
The outpud directory (../build/cutsPower__2023.08.18_06-59-25) is created!


In [31]:
tasks_to_run : [str] = []

print("Problems to work with:")

for rel_path in glob.iglob(problems_name_dir, recursive=True):
    print(rel_path)
    file_name: str = path.basename(rel_path)
    results_rel_path: str = "./" + out_dir_name + "/" + file_name + result_file_name_suff
    log_rel_path: str = "./" + out_dir_name + "/" + file_name + log_file_name_suff
    out_cuts_rel_path: str = "./" + out_dir_name + "/" + file_name + cuts_file_name_suff

    run_str: str = f"./CPlexCutsPower -i {rel_path} -o {log_rel_path} --comp_result {results_rel_path} --wcgn 1 --wcpl {cplex_workers} " \
    f"--iters {cuts_gen_rounds} --node_lim {node_lim} --model_cuts {model_cuts_switch} --trials_num {row_trials_num} --submat_limit {submat_size_lim} " \
    f"--cplex_tl {cplex_sec_tl} --cuts_tl {cuts_gen_sec_tl} --min_dim {min_dim} --max_dim {max_dim}"

    tasks_to_run.append(run_str)


../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz
../problems/benchmark/miplib2010_decomp2/instances/decomp2.mps.gz
../problems/benchmark/miplib2010_netdiversion/instances/netdiversion.mps.gz
../problems/benchmark/Andrea_Arias/instances/cmflsp50-24-8-8.mps.gz
../problems/benchmark/Michael_Bastubbe2/instances/graph20-20-1rand.mps.gz
../problems/benchmark/miplib2010_Robert_Fourer/instances/assign1-5-8.mps.gz
../problems/benchmark/miplib2010_sing/instances/sing326.mps.gz
../problems/benchmark/miplib2010_sing/instances/sing44.mps.gz
../problems/benchmark/miplib2010_50v-10/instances/50v-10.mps.gz
../problems/benchmark/miplib2010_roc/instances/rocII-5-11.mps.gz
../problems/benchmark/miplib2010_roc/instances/rocI-4-11.mps.gz
../problems/benchmark/miplib2010_lectsched-5-obj/instances/lectsched-5-obj.mps.gz
../problems/benchmark/miplib2010_csched008/instances/csched008.mps.gz
../problems/benchmark/miplib2010_rococoC10-001000/instances/rococoC10-001000.mps.gz
../problems/benchmar

In [32]:
print(f"There are {len(tasks_to_run)} tasks to run:\n")
for task_str in tasks_to_run:
    print(f"{task_str}\n")

There are 240 tasks to run:

./CPlexCutsPower -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.08.18_06-59-25/timtab1.mps.gz_log.txt --comp_result ./cutsPower__2023.08.18_06-59-25/timtab1.mps.gz_results.txt --wcgn 1 --wcpl 15 --iters 1 --node_lim 0 --model_cuts off --trials_num 7 --submat_limit 200 --cplex_tl 10800 --cuts_tl 0 --min_dim 100 --max_dim 10000

./CPlexCutsPower -i ../problems/benchmark/miplib2010_decomp2/instances/decomp2.mps.gz -o ./cutsPower__2023.08.18_06-59-25/decomp2.mps.gz_log.txt --comp_result ./cutsPower__2023.08.18_06-59-25/decomp2.mps.gz_results.txt --wcgn 1 --wcpl 15 --iters 1 --node_lim 0 --model_cuts off --trials_num 7 --submat_limit 200 --cplex_tl 10800 --cuts_tl 0 --min_dim 100 --max_dim 10000

./CPlexCutsPower -i ../problems/benchmark/miplib2010_netdiversion/instances/netdiversion.mps.gz -o ./cutsPower__2023.08.18_06-59-25/netdiversion.mps.gz_log.txt --comp_result ./cutsPower__2023.08.18_06-59-25/netdiversion.mps.gz_

In [ ]:
def run_task(run_str : str):
    # print(run_str)
    return subprocess.run(run_str, shell=True, cwd=run_path)

cpu_cores_to_use: int = min(available_cpu_cores, multiprocessing.cpu_count() - 1)

print(f"total cpu count is {multiprocessing.cpu_count()}, count of cpu cores to use is {cpu_cores_to_use}\n") 

starting_time = dt.now()
done_tasks_counter : int = 0

with multiprocessing.Pool(cpu_cores_to_use) as tasks_pool:
    for task_result in tasks_pool.imap_unordered(run_task, tasks_to_run):
        args: str = str(task_result.args)
        if task_result.returncode == 0:
            print(f"({done_tasks_counter}) The process is successfully finished!, args are:")
        else:
            print(f"({done_tasks_counter}) The process is failed!, return code is {task_result.returncode}, args are:")
        print(f"{args}\n")

        done_tasks_counter += 1
        if done_tasks_counter % 10 == 0:
            clear_output(wait=True)
            print(f"-------{done_tasks_counter} tasks are done, {len(tasks_to_run)-done_tasks_counter} rest-------\n")

In [ ]:
duration = dt.now() - starting_time
print(f"Work is finished!, duration is {duration}")